In [ ]:
import pandas as pd
import numpy as np

# 1 LOAD
RAW_FILE = "Library_raw.csv"

df_raw = pd.read_csv(RAW_FILE, header=0, skiprows=[1, 2], low_memory=False)

# Optional!?! keep full question text for reference
q_labels = pd.read_csv(RAW_FILE, header=None, nrows=3).iloc[1].to_dict()

# 2 RENAME COLUMNS
position_rename = {
    " ": "consent",
    "Q1": "library_engagement",
    " .1": "low_use_reason",
    " .2": "frequent_use_reason",
    "Q2": "essential_service",
    " .3": "digital_resources_secret",
    " .4": "study_space_value",
    " .5": "book_librarian_benefit",
    " .6": "other_service_text",
    " .7": "nonuse_reason",
    "Q3": "data_support_effectiveness",
    " .8": "data_support_unknown_reason",
    " .9": "data_access_difficulty",
    " .10": "data_needed_for_very_effective",
    "Q4": "most_compelling_metric",
    "Q5": "workflow_collaboration_story",
    "Q6": "healthcare_info_experience",
    " .11": "library_value_to_community",
    "Q7_NPS_GROUP": "dashboard_use_nps_group",
    "Q7": "dashboard_use_nps_score",
    " .12": "evidence_for_reports",
    " .13": "prefix",
    " .14": "name",
    " .15": "department",
    " .16": "role",
    "ResponseId": "response_id",
    "RecordedDate": "recorded_date",
    "Duration (in seconds)": "duration_seconds",
    "Progress": "progress_pct",
}

df = df_raw.rename(columns=position_rename).copy()

# 3 REMOVE TEST / INCOMPLETE RESPONSES
if "Status" in df.columns:
    df = df[df["Status"] != "Survey Preview"].copy()

if "Finished" in df.columns:
    df["Finished"] = df["Finished"].astype(str).str.upper().str.strip()
    df = df[df["Finished"].isin(["TRUE", "1"])].copy()

df = df.reset_index(drop=True)

# 4 DROP USELESS / SENSITIVE COLs
df = df.dropna(axis=1, how="all")

drop_cols = [
    "IPAddress", "RecipientLastName", "RecipientFirstName",
    "RecipientEmail", "ExternalReference",
    "LocationLatitude", "LocationLongitude",
    "DistributionChannel", "UserLanguage",
    "Q_RecaptchaScore", "Q_DuplicateRespondent"
]

df = df.drop(columns=[c for c in drop_cols if c in df.columns])

# 5 FIX COL TYPES
for col in ["StartDate", "EndDate", "recorded_date"]:
    if col in df.columns:
        df[col] = pd.to_datetime(df[col], errors="coerce")

if "duration_seconds" in df.columns:
    df["duration_seconds"] = pd.to_numeric(df["duration_seconds"], errors="coerce")

if "progress_pct" in df.columns:
    df["progress_pct"] = pd.to_numeric(df["progress_pct"], errors="coerce")

data_support_map = {
    "1 Not at all effectively": 1,
    "Not at all effectively": 1,
    "2 Slightly effectively": 2,
    "Slightly effectively": 2,
    "3 Moderately effectively": 3,
    "Moderately effectively": 3,
    "4 Effectively": 4,
    "Effectively": 4,
    "5 Very Effectively": 5,
    "Very Effectively": 5,
    "Unknown": 0,
    "1": 1, "2": 2, "3": 3, "4": 4, "5": 5,
}

if "data_support_effectiveness" in df.columns:
    df["data_support_effectiveness_num"] = df["data_support_effectiveness"].map(data_support_map)

if "dashboard_use_nps_score" in df.columns:
    df["dashboard_use_nps_score"] = pd.to_numeric(df["dashboard_use_nps_score"], errors="coerce")

# Check for stragglers
if "data_support_effectiveness" in df.columns and "data_support_effectiveness_num" in df.columns:
    missed = df[df["data_support_effectiveness"].notna() & df["data_support_effectiveness_num"].isna()]["data_support_effectiveness"].unique()
    if len(missed) > 0:
        print(f" Unmapped in data_support_effectiveness: {missed}")

# 6 HELPER COLs
if "dashboard_use_nps_score" in df.columns:
    def nps_bucket(score):
        if pd.isna(score):
            return np.nan
        elif score >= 9:
            return "Promoter"
        elif score >= 7:
            return "Passive"
        else:
            return "Detractor"

    df["dashboard_use_nps_group_calc"] = df["dashboard_use_nps_score"].apply(nps_bucket)

if "duration_seconds" in df.columns:
    df["duration_minutes"] = (df["duration_seconds"] / 60).round(1)

if "recorded_date" in df.columns:
    df["survey_month"] = df["recorded_date"].dt.to_period("M").astype(str)

# 7 SPLIT DATA
quant_cols = [
    "response_id", "recorded_date", "survey_month",
    "duration_minutes", "progress_pct",
    "data_support_effectiveness_num",
    "dashboard_use_nps_score",
    "dashboard_use_nps_group",
    "dashboard_use_nps_group_calc",
    "department", "role"
]
quant_cols = [c for c in quant_cols if c in df.columns]
df_quant = df[quant_cols].copy()

text_cols = [
    "response_id",
    "low_use_reason", "frequent_use_reason",
    "digital_resources_secret", "study_space_value",
    "book_librarian_benefit", "other_service_text", "nonuse_reason",
    "data_support_unknown_reason", "data_access_difficulty",
    "data_needed_for_very_effective",
    "most_compelling_metric",
    "workflow_collaboration_story",
    "healthcare_info_experience",
    "library_value_to_community",
    "evidence_for_reports"
]
text_cols = [c for c in text_cols if c in df.columns]
df_text = df[text_cols].copy()

# 8 EXPORT
df.to_csv("admin_clean_full.csv", index=False)
df_quant.to_csv("admin_quant.csv", index=False)
df_text.to_csv("admin_text.csv", index=False)

print(f"✓ Cleaned rows: {len(df)}")
print(f"✓ Quant columns: {len(df_quant.columns)}")
print(f"✓ Text columns: {len(df_text.columns)}")

if len(df) == 0:
    print("WARNING: No rows survived filtering. Check Status and Finished.")
else:
    print("\nSample stats for numeric fields:")
    rating_cols = [
        c for c in [
            "data_support_effectiveness_num",
            "dashboard_use_nps_score"
        ] if c in df.columns
    ]
    if rating_cols:
        print(df[rating_cols].describe().round(2))

print(df[[
    "data_support_effectiveness",
    "data_support_effectiveness_num"
]].head())

✓ Cleaned rows: 7
✓ Quant columns: 9
✓ Text columns: 9

Sample stats for numeric fields:
       data_support_effectiveness_num  dashboard_use_nps_score
count                             5.0                     5.00
mean                              3.2                     5.00
std                               1.1                     3.61
min                               2.0                     0.00
25%                               3.0                     4.00
50%                               3.0                     5.00
75%                               3.0                     6.00
max                               5.0                    10.00
  data_support_effectiveness  data_support_effectiveness_num
0   3 Moderately effectively                             3.0
1                        NaN                             NaN
2                        NaN                             NaN
3         5 Very Effectively                             5.0
4   3 Moderately effectively           